In [ ]:
import zipfile
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

In [ ]:
sq_data_l = pd.read_pickle('data_ising_square_largeL.pkl')

In [ ]:
for key in sq_data_l:
    print(f"{key}: type = {type(sq_data_l[key])}")

In [ ]:
for key in sq_data_l[8]:
    print(f"{key}: type ={type(sq_data_l[8][key])}")

In [ ]:
dynamic_vars = {}

for i in sq_data_l.keys():
    if isinstance(sq_data_l[i], dict):
        for j in sq_data_l[i].keys():
            key = f"{j}_{i}"
            dynamic_vars[key] = sq_data_l[i][j]
    elif isinstance(sq_data_l[i], list):
        for idx, val in enumerate(sq_data_l[i]):
            key = f"{idx}_{i}"
            dynamic_vars[key] = val
    else:
        # if it's neither dict nor list, you can decide what to do
        print(f"Skipping key {i}: unexpected type {type(sq_data_l[i])}")

In [ ]:
len(dynamic_vars.keys())

In [ ]:
dynamic_vars.keys()

In [ ]:
plt.figure(figsize=(15,15))
Tc_l = []
Tc_chi = []
for i in sq_data_l['Ls']:
    pos =np.argmax(dynamic_vars[f'C_{i}'][:,0])
    pos_chi = np.argmax(dynamic_vars[f'chi_{i}'][:,0])
    tc =dynamic_vars[f'Ts_{i}'][pos]
    tc2 =dynamic_vars[f'Ts_{i}'][pos_chi]
    print(fr'critical temp for L = {i}=>{tc:.3f}')
    Tc_l.append(tc)
    Tc_chi.append(tc2)
    
    plt.subplot(2,2,1)
    plt.errorbar(dynamic_vars[f'Ts_{i}'],dynamic_vars[f'C_{i}'][:,0],dynamic_vars[f'C_{i}'][:,1], label=f'L = {i}, Tc_{i}={tc: .2f}')
    plt.axvline(dynamic_vars['Ts_8'][pos], linestyle =':')

    plt.subplot(2,2,3)
    plt.errorbar(dynamic_vars[f'Ts_{i}'],dynamic_vars[f'E_{i}'][:,0],dynamic_vars[f'E_{i}'][:,1], label=f'L = {i}, Tc_{i}={tc: .2f}')
    plt.axvline(dynamic_vars['Ts_8'][pos], linestyle =':')
    
    plt.subplot(2,2,2)
    plt.errorbar(dynamic_vars[f'Ts_{i}'],dynamic_vars[f'chi_{i}'][:,0],dynamic_vars[f'chi_{i}'][:,1], label=f'L = {i},Tc_{i}={tc: .2f} ')
    plt.axvline(dynamic_vars['Ts_8'][pos], linestyle =':')

    plt.subplot(2,2,4)
    plt.errorbar(dynamic_vars[f'Ts_{i}'],dynamic_vars[f'UB_{i}'][:,0],dynamic_vars[f'UB_{i}'][:,1], label=f'L = {i}')
   # plt.axvline(dynamic_vars['Ts_8'][pos], linestyle =':', label = f'Tc_{i}={tc: .2f}')
    
Tc = 2.26

plt.subplot(2,2,1)
plt.axvline(Tc,linestyle = ':', c='r', label = 'Tc')
plt.title(r'$C_V\;V_S\;T$ ')
plt.xlabel(r'Temp $\rightarrow$')
plt.ylabel(r'$C_V\rightarrow$')
plt.legend()

plt.subplot(2,2,2)
plt.axvline(Tc,linestyle = ':', c='r', label = 'Tc')
plt.title(r'$\chi\;V_S\;T$ ')
plt.xlabel(r'Temp $\rightarrow$')
plt.ylabel(r'$\chi \rightarrow$')
plt.legend()

plt.subplot(2,2,3)
plt.axvline(Tc,linestyle = ':', c='r', label = 'Tc')
plt.title(r'$E\;V_S\;T$ ')
plt.xlabel(r'Temp $\rightarrow$')
plt.ylabel(r'$E \rightarrow$')
plt.legend()

plt.subplot(2,2,4)
plt.axvline(Tc,linestyle = ':', c='r', label = 'Tc')
plt.title(r'$UB\;V_S\;T$ ')
plt.xlabel(r'Temp $\rightarrow$')
plt.ylabel(r'$UB \rightarrow$')
plt.axvline(2.269,linestyle = ':', c='b', label = 'cross pt=2.269')
plt.legend()
plt.xlim(2.24,2.35)
plt.tight_layout()

plt.show()



In [ ]:
L_vals = np.array(sq_data_l['Ls'])
L_inv = 1/L_vals 
fit_coeff = np.polyfit(L_inv, Tc_chi, 1)
plt.plot([0, 1./8], np.polyval(fit_coeff, [0, 1./8]), label=r'fit_$\chi$')
fit_coeff2 = np.polyfit(L_inv, Tc_l, 1)
plt.plot([0, 1./8], np.polyval(fit_coeff2, [0, 1./8]), label=r'fit_$C_V$')
plt.plot(L_inv, Tc_l, 'o', c='orange', linewidth = 2, label = '$C_V$')
plt.plot(L_inv, Tc_chi,  'o',c='b', linewidth = 2, label = r'$\chi$')
plt.axhline(Tc, linestyle=':', c='r',linewidth =2, label= f'Tc = {Tc}')

plt.xlabel(r'$\frac{1.}{L} \; \rightarrow$')
plt.ylabel(r'$critical\,temp\,(T_C)\; \rightarrow$')

plt.legend()
plt.show()



In [ ]:
L_inv

In [ ]:
Tc_chi

In [ ]:
tc_chi = np.polyval(fit_coeff,0)
tc_c = np.polyval(fit_coeff2,0)
Tc_f = 0.5*(tc_c+tc_chi)
print('Tc=',Tc_f)

In [ ]:
chi = []
for l in sq_data_l['Ls']:
    chi.append(np.max(dynamic_vars[f'chi_{l}']))

plt.plot(sq_data_l['Ls'],chi,'o-')

In [ ]:
t_c = []
target_Tc = 2.267
delta = 0.2  

for l in sq_data_l['Ls']:
    if 2 * l in sq_data_l['Ls']:
        l_2 = 2 * l
    else:
        l_2 = int(l/2)

    diff = dynamic_vars[f'UB_{l_2}'][:, 0] - dynamic_vars[f'UB_{l}'][:, 0]
    a = np.where(np.diff(np.sign(diff)) != 0)[0]

    if a.size > 0:
       
        candidate_Ts = dynamic_vars[f'Ts_{l}'][a]
       
        mask = np.abs(candidate_Ts - target_Tc) < delta
        if np.any(mask):
            
            idx = a[mask][0]
            t_c.append(dynamic_vars[f'Ts_{l}'][idx])
        else:
            print(f"No crossing near Tc=2.27 for L={l}, appending NaN")
            t_c.append(np.nan)
    else:
        print(f"No crossing found for L={l}, appending NaN")
        t_c.append(np.nan)

t_c

In [ ]:
t_c4 = []
target_Tc = 2.267
delta = 0.2  

for l in sq_data_l['Ls']:
    if 4 * l in sq_data_l['Ls']:
        l_2 =  l*4
    else:
        l_2 = int(l/4)

    diff = dynamic_vars[f'UB_{l_2}'][:, 0] - dynamic_vars[f'UB_{l}'][:, 0]
    a = np.where(np.diff(np.sign(diff)) != 0)[0]

    if a.size > 0:
      
        candidate_Ts = dynamic_vars[f'Ts_{l}'][a]
      
        mask = np.abs(candidate_Ts - target_Tc) < delta
        if np.any(mask):
          
            idx = a[mask][0]
            t_c4.append(dynamic_vars[f'Ts_{l}'][idx])
        else:
            print(f"No crossing near Tc=2.27 for L={l}, appending NaN")
            t_c4.append(np.nan)
    else:
        print(f"No crossing found for L={l}, appending NaN")
        t_c4.append(np.nan)

t_c4

In [ ]:
Ls = sq_data_l['Ls']

In [ ]:
chi_max_values=[]
for l in Ls:
    chi_max_values.append(np.max(dynamic_vars[f'chi_{l}']))
logL = np.log(Ls)
log_chi = np.log(chi_max_values)

slope_gamma, intercept = np.polyfit(logL, log_chi, 1)
print(f"γ/ν ≈ {slope_gamma:.3f}")

In [ ]:
chi_max_values

In [ ]:
C_max_values=[]
for l in Ls:
    C_max_values.append(np.max(dynamic_vars[f'C_{l}']))
logL = np.log(Ls)
log_chi = np.log(C_max_values)

slope_alpha, intercept = np.polyfit(logL, log_chi, 1)
print(f"𝛼/ν ≈ {slope_alpha:.3f}")

In [ ]:
Ls = sq_data_l['Ls'] 

data = {}
for L in Ls:
    Ts = np.array(dynamic_vars[f'Ts_{L}'])
    C = np.array(dynamic_vars[f'C_{L}'][:, 0]) 
    data[L] = {'T': Ts, 'C': C}


def scaling_collapse_C(data, Ls, Tc, nu, alpha):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8,6))

    for L in Ls:
        T = data[L]['T']
        C = data[L]['C']

        x = (T - Tc) * L**(1/nu)
        y = C / L**(alpha/nu)

        plt.plot(x, y, 'o-', label=f"L={L}")

    plt.xlabel(r"$(T - T_c) L^{1/\nu}$", fontsize=14)
    plt.ylabel(r"$C / L^{\alpha/\nu}$", fontsize=14)
    plt.title(f"Collapse: $T_c$={Tc:.3f}, $\\nu$={nu:.2f}, $\\alpha$={alpha:.2f}")
    plt.grid(True)
    plt.legend()
    plt.xlim(-5,5)
    plt.show()
scaling_collapse_C(data, Ls, Tc=2.27, nu=1.0, alpha=0.261)


In [ ]:
Ls = sq_data_l['Ls'] 

data = {}
for L in Ls:
    Ts = np.array(dynamic_vars[f'Ts_{L}'])
    chi = np.array(dynamic_vars[f'chi_{L}'][:, 0])  
    data[L] = {'T': Ts, 'chi': chi}


def scaling_collapse_chi(data, Ls, Tc, nu, gamma):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8,6))

    for L in Ls:
        T = data[L]['T']
        chi = data[L]['chi']

        x = ((T - Tc)/Tc) * L**(1/nu)
        y = chi / L**(gamma/nu)

        plt.plot(x, y, 'o-', label=f"L={L}")

    plt.xlabel(r"$(T - T_c) L^{1/\nu}$", fontsize=14)
    plt.ylabel(r"$\chi / L^{\gamma/\nu}$", fontsize=14)
    plt.title(f"Collapse: $T_c$={Tc:.2f}, $\\nu$={nu:.2f}, $\\gamma$={gamma:.3f}")
    plt.grid(True)
    plt.legend()
    plt.xlim(-2.5,7)
    plt.show()
scaling_collapse_chi(data, Ls, Tc=2.27, nu=1.0, gamma=1.763)
